# 10 · Práctica guiada · Carteras de inversión

**Duración estimada:** 1h 30min  ·  Individual

En esta práctica vamos a:

1. Ver qué es una **cartera de inversión** y por qué se diversifica.
2. Descargar precios de acciones reales con `yfinance`.
3. Calcular **rentabilidades, riesgo y ratio de Sharpe**.
4. Componer una cartera con pesos a medida.
5. **RETO final:** generar miles de carteras aleatorias y encontrar la mejor.

💡 A lo largo del notebook encontrarás **tips** (así: 💡). Si te atascas, léelos. Si no, intenta primero por tu cuenta.

---

## 1 · ¿Qué es una cartera de inversión?

Una **cartera** es un conjunto de activos (acciones, bonos, etc.) en los que invertimos. A cada activo le asignamos un **peso** (qué porcentaje del dinero total va a cada uno).

**Conceptos clave:**

- **Pesos** `w₁, w₂, …, w_K` → cuánto invertimos en cada activo. Suman 1 (el 100% del capital).
- **Rentabilidad esperada** → cuánto esperamos ganar (de media) en un periodo.
- **Riesgo (volatilidad)** → cómo de "agitada" es la rentabilidad (medida con la desviación típica).
- **Diversificación** → repartir entre varios activos para que el conjunto sea menos volátil que cada activo por separado.

**Idea intuitiva:** si pones todo en una acción y esa acción cae, lo pierdes todo. Si repartes entre 5, el conjunto se mueve menos (aunque tampoco sube tanto si una acción sube mucho).

---

## 2 · Descargamos los datos

Usaremos 5 acciones españolas del IBEX 35, las mismas que ya viste en el notebook anterior.

💡 **Tip:** si yfinance no está instalado, descomenta la primera línea:

In [ ]:
# !pip install yfinance -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

### Paso 1 · Descarga los precios de cierre ajustados

In [ ]:
# Definimos los tickers y el rango de fechas
tickers = ['IBE.MC', 'ITX.MC', 'SAN.MC', 'BBVA.MC', 'TEF.MC']
nombres = ['Iberdrola', 'Inditex', 'Santander', 'BBVA', 'Telefónica']
start   = '2021-01-01'
end     = '2024-12-31'

# Descargamos los precios de cierre ajustados
# (auto_adjust=True aplica dividendos y splits)
data = yf.download(tickers, start=start, end=end, auto_adjust=True)['Close']
data.columns = nombres
data.head()

### Paso 2 · Echa un vistazo a los datos

In [ ]:
# 💡 Tip: comprueba el tamaño (`shape`), si hay nulos (`isna().sum()`) y las primeras/últimas filas.

### Paso 3 · Visualiza la evolución de los precios (normalizados)

Para comparar visualmente, normalizamos: dividimos cada precio por el primero de la serie. Así todos empiezan en 1.

In [ ]:
norm = data / data.iloc[0]
norm.plot(figsize=(11, 5), linewidth=1.5)
plt.ylabel('Precio normalizado (1 = inicio)')
plt.title('Evolución de las 5 acciones (base 1)')
plt.grid(alpha=0.3)
plt.show()

---

## 3 · De precios a rentabilidades

En finanzas no trabajamos con precios sino con **rentabilidades**: cuánto ha subido (o bajado) el precio de un día para otro.

**Rentabilidad simple diaria:**

$$r_t = \frac{P_t - P_{t-1}}{P_{t-1}}$$

💡 **Por qué rentabilidades y no precios:** las rentabilidades se pueden sumar (la rentabilidad de 2 días es la suma de las 2 diarias), son aproximadamente simétricas y son comparables entre acciones de precios muy distintos.

### Paso 4 · Calcula la rentabilidad diaria

In [ ]:
# Rentabilidad diaria: (P_t - P_{t-1}) / P_{t-1}
# pct_change() lo hace en una línea. La primera fila es NaN (no hay día anterior).
retornos = data.pct_change().dropna()
retornos.head()

### Paso 5 · Visualiza la distribución de rentabilidades de Iberdrola

In [ ]:
# Lo hacemos con Iberdrola como ejemplo
data['Iberdrola'].pct_change().dropna().hist(bins=50, edgecolor='k')
plt.xlabel('Rentabilidad diaria')
plt.ylabel('Frecuencia')
plt.title('Distribución de rentabilidades diarias — Iberdrola')
plt.axvline(0, color='red', linestyle='--')
plt.show()

💡 **Observa:** la distribución está centrada cerca de 0, es aproximadamente simétrica pero tiene una **cola a la izquierda** (caídas fuertes puntuales) y otra a la derecha (subidas fuertes). Esos valores extremos son los que vimos en la sesión de outliers.

---

## 4 · Una sola acción: retorno y riesgo

Para una acción, el **retorno medio** y la **volatilidad** se calculan con la media y la desviación típica de sus rentabilidades diarias.

💡 **Anualización:** los mercados abren ~252 días al año. Para comparar acciones en la misma escala temporal, anualizamos:

- Retorno anual ≈ retorno diario medio × 252
- Volatilidad anual ≈ volatilidad diaria × √252

### Paso 6 · Calcula el retorno anualizado y la volatilidad anualizada de cada acción

In [ ]:
# 💡 Tip:
# - retorno_diario = retornos.mean()
# - volatilidad_diaria = retornos.std()
# - retorno_anual = retorno_diario * 252
# - volatilidad_anual = volatilidad_diaria * np.sqrt(252)
#

---

## 5 · Componemos una cartera

Una **cartera** es un vector de pesos `w` (uno por acción). Por ejemplo, con 5 acciones, `w = [0.2, 0.2, 0.2, 0.2, 0.2]` es una cartera **equitativa** (mismo peso para todas). `w.sum()` debe ser 1.

**Métricas de la cartera:**

- **Retorno esperado** = `w · μ` (producto escalar del vector de pesos por el vector de rentabilidades medias).
- **Volatilidad** = `√(w · Σ · w)` donde `Σ` es la **matriz de covarianzas** de las rentabilidades. La matriz de covarianzas ya la calcula pandas con `retornos.cov()`.
- **Ratio de Sharpe** = retorno / volatilidad (mayor = mejor relación riesgo/beneficio, sin ajustar por tipo libre de riesgo en esta práctica).

### Paso 7 · Define los pesos y calcula retorno, volatilidad y Sharpe de una cartera equitativa

In [ ]:
# Pesos equitativos: 1/5 para cada acción
w = np.array([0.2, 0.2, 0.2, 0.2, 0.2])
print('Suma de pesos:', w.sum())  # debe ser 1

In [ ]:
# Vector de rentabilidades medias diarias de las 5 acciones
mu = retornos.mean()       # Serie de pandas, una media por acción
print(mu)

In [ ]:
# Retorno de la cartera: producto escalar w @ mu
retorno_cartera = w @ mu
print(f'Retorno diario   : {retorno_cartera:.6f}')
print(f'Retorno anual    : {retorno_cartera * 252:.4f}')

In [ ]:
# Matriz de covarianzas (diaria) y volatilidad de la cartera
cov = retornos.cov()
volatilidad_cartera = np.sqrt(w @ cov @ w)
print(f'Volatilidad diaria: {volatilidad_cartera:.6f}')
print(f'Volatilidad anual : {volatilidad_cartera * np.sqrt(252):.4f}')

In [ ]:
# Ratio de Sharpe (anualizado)
sharpe = (retorno_cartera * 252) / (volatilidad_cartera * np.sqrt(252))
print(f'Sharpe: {sharpe:.3f}')

---

## 6 · Probamos varios pesos manualmente

Ahora vamos a ver cómo cambian las métricas según los pesos. Crea una pequeña tabla con 3 o 4 carteras (por ejemplo: equitativa, todo en Iberdrola, todo en Inditex, y una mixta) y compara.

### Paso 8 · Crea y compara 4 carteras

In [ ]:
# 💡 Tip: define los 4 vectores de pesos, calcula retorno/vol/Sharpe de cada uno con las mismas fórmulas, y monta un DataFrame resumen.

---

## 7 · 🏁 RETO: carteras aleatorias

**Objetivo:** generar **1000 carteras aleatorias**, calcular sus métricas y visualizar el resultado en el espacio **riesgo (volatilidad) vs retorno**, coloreando por Sharpe. Encontrar la cartera con **mayor Sharpe** y mostrar sus pesos.

Esto es lo que hacen los profesionales como primer paso para entender la estructura de oportunidades de inversión.

### ¿Cómo se genera una cartera aleatoria?

1. Generar `K` números aleatorios entre 0 y 1 (siendo `K` el número de acciones).
2. Normalizarlos para que sumen 1 (dividir cada uno por la suma total).
3. Ya tienes una cartera aleatoria.

💡 **Tip 1:** `np.random.rand(K)` genera K números aleatorios entre 0 y 1.

💡 **Tip 2:** para que sumen 1, divide por la suma: `w / w.sum()`.

💡 **Tip 3:** haz un bucle `for` que en cada iteración genere una cartera, calcule retorno/vol/Sharpe, y guarde los resultados en una lista.

💡 **Tip 4:** al final convierte la lista en un DataFrame para poder buscar la mejor (`idxmax` del Sharpe).

### Paso 9 · Genera 1000 carteras aleatorias y almacena sus métricas

In [ ]:
# # Tu código aquí

### Paso 10 · Visualiza: scatter riesgo vs retorno, color por Sharpe

💡 **Tip:** el scatter más útil en finanzas: eje x = volatilidad anual, eje y = retorno anual, color = Sharpe. Usa `plt.scatter(x, y, c=sharpe, cmap='viridis')` y `plt.colorbar(label='Sharpe')`.

In [ ]:
# # Tu código aquí

### Paso 11 · Encuentra la mejor cartera (mayor Sharpe)

💡 **Tip:** una vez tienes el DataFrame con las 1000 carteras, `df.loc[df['sharpe'].idxmax()]` te da la fila ganadora. Muestra los pesos, el retorno, la volatilidad y el Sharpe de esa cartera.

In [ ]:
# # Tu código aquí

### Paso 12 · Bonus · Dibuja la mejor cartera sobre el scatter

💡 **Tip:** usa `plt.scatter(...)` otra vez con `color='red'` y `s=200` para que se vea grande, y pon una etiqueta con `plt.annotate(...)`.

In [ ]:
# # Tu código aquí

---

## Cierre

**Lo que has hecho hoy:**

- Descargado datos reales con `yfinance` y visto que las acciones se mueven de forma muy distinta.
- Calculado rentabilidades, retorno esperado, volatilidad y ratio de Sharpe.
- Comprobado que una cartera mixta se mueve "entre" sus componentes.
- Generado 1000 carteras aleatorias y encontrado la de mejor Sharpe visualmente.

**Lo que NO hemos visto y queda para el futuro:**

- **Frontera eficiente:** la curva que une las carteras con mejor retorno para cada nivel de riesgo. Es lo que viene después de este scatter.
- **Optimización con `scipy`:** en vez de generar 1000 al azar, encontrar la mejor matemáticamente.
- **Tipo libre de riesgo y Sharpe "real":** aquí usamos Sharpe = retorno / volatilidad; el Sharpe clásico resta la rentabilidad del activo sin riesgo.
- **Backtesting:** ver cómo se habría comportado una estrategia en el pasado.
- **Correlaciones y diversificación real:** la matriz de covarianzas ya la has visto, pero hay mucho más detrás.